# 7.5 优化步骤②：CV 流水与切块

均衡性解决「核间」的空等；本节解决**引擎间**的空等——Cube 与 Vector 一来一回互相等待。

## 1. CV 流水：四拍的重叠账

第 5 章已经建立了三种编排，这里从性能视角复述其账单：

| 编排 | Vector 视角 | Cube 视角 | 剩余浪费 |
|--|--|--|--|
| 串行 | 等 Cube 全部做完 | 等 Vector 全部做完 | 全部空等 |
| 乒乓（NBuffer） | 搬运与计算重叠 | 等 V1 | Cube/Vector 仍串行 |
| Preload | 被 C1(i+1)/C2(i) 掩盖 | 被 V1 掩盖 | 收尾空拍、尾块半拍 |

FA 的时间画像 **V1 > C1 ≈ C2 >> V2**（softmax 及其前后处理远长于两次矩阵乘）说明 FA 通常是 **Vector bound**：优化 Cube 收益有限，真正的空间在：

1. **流水编排**：用两个 Cube 拍（i 的 C2 与 i+1 的 C1）掩盖一个长 V1——preload 的本质；
2. **缩短 V1 本身**：把 softmax 从普通 Vector 指令迁移到 **VF**（Vector Function，见 7.6 节与第 8 章）；
3. **配平 CV 比例**：AIC:AIV = 1:2 的硬件上，nRatio 等参数控制 Cube/Vector 的任务分配比例，让两侧同时干完。

## 2. 切块：块大小与计算强度

7.2 节的公式 `I = BaseM·BaseN / ((BaseM+BaseN)·D_size)` 给出切块的性能判据：

- 块越大 → I 越高 → 越接近计算受限平台（好）；
- 但块大小受 **UB 容量**约束（Q/K/V/P + m/l 全要装下），且大块加剧 **尾块浪费**（形状不满一块时按整块算，第 5 章 5.8 节）；
- S2 方向的切块还与 softmax 的归约粒度绑定：S2 块是 m/l 增量更新的单元。

**切块是 UB 容量、计算强度、尾块浪费、流水节拍四者的联合优化**——这也是 Tiling 设计的核心艺术（第 3 章原则一的完整含义）。

## 3. 实例：Softmax VF 的切块实践

以 FA softmax（V1 拍）的 VF 实现为例，看切块如何落到代码：

**寄存器宽度定粒度**：Vector 核寄存器大小 256B，一条 VF 指令处理 **64 个 FP32**。按大融合在 S2 方向的有效宽度（originN）与是否首次 softmax 计算，分派为函数族：

| 函数 | 条件 |
|--|--|
| ProcessVec1NoUpdate / ProcessVec1Update 主入口 | 第一次计算（无 m/l 可更新） / 后续轮次 |
| └ Imp164 | originN ≤ 64（一条指令吃下） |
| └ Impl128 | originN = 128（整块对齐，最快路径） |
| └ GeneralImpl128 / GeneralImpl256 | 64 < originN < 128 / 128 < originN ≤ 256（通用带尾块） |

**两个对齐细节**：

- AIC:AIV = 1:2 时，mm1 的输出按行均分到两个 AIV 做 Vec1，均分后的行数向上对齐到 S1BaseSize；
- originN（S2 有效列数）向上对齐到 256B 的整数倍——FP32 4 字节下即 64 的整数倍，天然匹配寄存器宽度；但不一定是 s2BaseSize 的整数倍，所以尾块走 GeneralImpl 路径。

**走 Dn（数据不搬）的判定**：无位置编码 pse、无 attention mask、无 drop mask、s1BaseSize ≠ 64、模板对齐 128、无 rope——满足全部条件才走 Dn 省掉搬运。S2 切分大时用 Dn、S2 小时 ND 更好。

> 结论：**「每条指令的形状」本身就是切块设计**——函数族的存在说明工程上按 originN 的落点分派专门路径，让每种形状都吃到对齐红利。

## 小测验

1. FA 的时间画像 V1 > C1 ≈ C2 >> V2 意味着什么 bound？三条优化路径分别是什么？
2. 写出块大小与计算强度的关系。块大小的四个约束是什么？
3. 一条 VF 指令处理多少个 FP32？为什么 originN 要对齐到 256B 的整数倍？
4. Softmax VF 函数族按哪两个维度分派？NoUpdate 与 Update 的区别是什么？